# Jalachakra: Baseline Model Evaluation

This notebook independently verifies and evaluates the existing `reduced_input/v1` model. It does not retrain a model or treat synthetic/derived labels as field observations.

**Environment:** use Python 3.12 with scikit-learn 1.7.1 to load the saved pipeline. The artifact is version-sensitive; a newer scikit-learn can fail during prediction. Install the project ML requirements, then pin `scikit-learn==1.7.1`, as described in `notebooks/03_cwr_baseline_and_ml_demo.ipynb`.

**Research target:** `etc_mm_day = ET0 x Kc`, in mm/day. It is a reference-derived estimate from FAO-56 Penman-Monteith ET0 and the project's crop-stage Kc curves, not measured field water use.

**Primary source:** `Climate_Data.csv`, the project's daily weather series. Crop-stage reference files define Kc. District soil/rainfall, crop recommendation, and season-context datasets are supporting resources and are not joined as model predictors.

The supplied `crop_water_data.csv -> water_requirement_liters` was rejected as ground truth in the existing audit. See `docs/target_assessment.md` and `docs/dataset_analysis.md` section 5.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / 'ml').is_dir() and (path / 'datasets').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from within the Jalachakra project.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PANEL_PATH = PROJECT_ROOT / 'datasets' / 'processed' / 'cwr_baseline_panel.parquet'
METADATA_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'reduced_input' / 'v1' / 'metadata.json'
PIPELINE_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'reduced_input' / 'v1' / 'pipeline.joblib'
panel = pd.read_parquet(PANEL_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
print(f'Panel: {panel.shape[0]:,} rows x {panel.shape[1]} columns')
print(f'Dates: {pd.to_datetime(panel.date).min().date()} to {pd.to_datetime(panel.date).max().date()}')
print(f'Crops: {sorted(panel.crop.unique())}')
print(f'Target candidate: {metadata["target_definition"]["name"]} ({metadata["target_definition"]["unit"]})')

Panel: 158,118 rows x 35 columns
Dates: 2018-01-01 to 2024-12-29
Crops: ['chickpea', 'cotton', 'cumin', 'finger_millet', 'groundnut', 'maize', 'paddy', 'wheat']
Target candidate: etc_mm_day (mm/day)


## Target and leakage audit

The estimation model uses only weather, recent rainfall, calendar, crop and growth-stage predictors available at the observation date. ET0, Kc, ETc and irrigation quantities derived from them are excluded: including them would leak the reference-derived target. The panel's rolling rainfall features are trailing windows, not centered windows.

In [2]:
from ml import target_builder as tb

task = tb.TASKS['reduced_input']
target = tb.TARGET_COLUMN
assert target == 'etc_mm_day'
assert metadata['target_definition']['name'] == target
assert metadata['target_definition']['definition'] == 'ETc = ET0 x Kc'
assert metadata['target_definition']['provenance_class'] == 'reference_derived'
assert metadata['target_definition']['is_field_observed'] is False
assert list(task.features) == metadata['feature_order']
assert not (set(task.features) & tb.LEAKAGE_SENSITIVE), 'Target leakage detected'
assert panel[target].notna().all() and (panel[target] >= 0).all()
print('Target provenance verified: reference-derived, not field-observed.')
print(f'Feature count: {len(task.features)}; no target-derived leakage features.')
print('Predictors:', ', '.join(task.features))

Target provenance verified: reference-derived, not field-observed.
Feature count: 16; no target-derived leakage features.
Predictors: t2m, t2m_min, t2m_max, rh2m, temp_range_c, rainfall_mm, rain_sum_7d_mm, rain_sum_30d_mm, days_since_rain, day_of_year, doy_sin, doy_cos, month, days_after_sowing, crop, growth_stage


## Chronological evaluation

A time-ordered split is required because weather is autocorrelated and crop cohorts overlap in time. The existing experiment fits on the earliest 60%, calibrates on the next 20%, and evaluates once on the final 20%. The calibration segment is not reused as the test set.

In [ ]:
features, labels, dates = tb.build_task_frame(panel, task)
split = tb.chronological_split(dates)
test_mask = split['test']
x_test = features.loc[test_mask]
y_test = labels.loc[test_mask].to_numpy(dtype=float)
split_metadata = metadata['split']
assert int(split['train'].sum()) == split_metadata['train_rows']
assert int(split['validation'].sum()) == split_metadata['calibration_rows']
assert int(test_mask.sum()) == split_metadata['test_rows']
assert str(pd.Timestamp(split['train_cutoff']).date()) == split_metadata['train_end']
assert str(pd.Timestamp(split['validation_cutoff']).date()) == split_metadata['calibration_end']
assert str(pd.to_datetime(dates).max().date()) == split_metadata['test_end']
print(f"Train through {split_metadata['train_end']}: {split_metadata['train_rows']:,} rows")
print(f"Calibration through {split_metadata['calibration_end']}: {split_metadata['calibration_rows']:,} rows")
print(f"Held-out test through {split_metadata['test_end']}: {len(y_test):,} rows")

## Existing baseline model and held-out metrics

The saved artifact is an XGBoost regressor selected after comparison with gradient boosting, random forest, ridge and a mean baseline on the same chronological partitions. The cell below loads that artifact and recomputes test metrics on the held-out rows; it does not fit or modify a model.

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=UserWarning)
    pipeline = joblib.load(PIPELINE_PATH)
prediction = np.asarray(pipeline.predict(x_test), dtype=float)
computed = {
    'mae_mm_day': mean_absolute_error(y_test, prediction),
    'rmse_mm_day': float(np.sqrt(mean_squared_error(y_test, prediction))),
    'r2': r2_score(y_test, prediction),
}
stored = metadata['metrics']['test']
for metric, value in computed.items():
    assert np.isclose(value, stored[metric], rtol=1e-3, atol=1e-4), (metric, value, stored[metric])
print('Held-out chronological test metrics (mm/day where applicable)')
print(pd.Series(computed).to_string(float_format=lambda value: f'{value:.4f}'))
print(f'\nTest rows: {len(y_test):,}; model: {metadata["algorithm"]}')
print('Metrics independently recomputed from the saved pipeline and test rows; artifact values agree.')

In [ ]:
import matplotlib.pyplot as plt

test_dates = pd.to_datetime(dates.loc[test_mask]).reset_index(drop=True)
observed = pd.Series(y_test, name='Reference-derived ETc')
predicted = pd.Series(prediction, name='Model estimate')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(observed, predicted, s=8, alpha=0.2, color='#167d72', edgecolors='none')
lower = min(observed.min(), predicted.min())
upper = max(observed.max(), predicted.max())
axes[0].plot([lower, upper], [lower, upper], color='#9b4a32', linestyle='--', linewidth=1)
axes[0].set(xlabel='Reference-derived ETc (mm/day)', ylabel='Predicted ETc (mm/day)', title='Held-out chronological test')

monthly = pd.DataFrame({'date': test_dates, 'absolute_error': np.abs(y_test - prediction)})
monthly['month'] = monthly.date.dt.to_period('M')
monthly_mae = monthly.groupby('month', observed=True).absolute_error.mean()
axes[1].plot(monthly_mae.index.astype(str), monthly_mae.values, color='#416a9c', marker='o', markersize=3)
axes[1].set(xlabel='Test month', ylabel='MAE (mm/day)', title='Monthly held-out error')
axes[1].tick_params(axis='x', labelrotation=60)
fig.tight_layout()
plt.show()

## Interpretation and limits

Report the result as model estimation of a **reference-derived crop water demand**, not measured CWR or observed irrigation use. The weather series represents one unresolved location; geographic generalization is not established. Crop stages and Kc curves include documented assumptions. The test evaluates agreement with the FAO-56/Kc reference, not field-level water consumption.

The existing artifact reports test MAE 0.3687 mm/day, RMSE 0.5105 mm/day and R² 0.9527 over 31,560 rows. Split-conformal intervals were calibrated on a separate time segment and achieved 0.9014 empirical coverage at nominal 0.90. These are saved experiment results and are asserted against recomputation above; they are not claims of field-observed accuracy.